# 🍽️ FoodWise AI – Online Food Ordering & Delivery Analytics

**IBM SkillsBuild Internship Project**

---

## Project Overview

This notebook presents an end-to-end data analytics and machine learning pipeline for understanding online food delivery behaviour in Bengaluru, India.

**Dataset:** 388 customer records with 14 features including demographics, income, education, family size, customer type, location coordinates, and ordering feedback.

**Business Objective:** Identify the key drivers that influence whether a customer will order food online (`Output = Yes/No`) and build a predictive model to classify future customers.

---

### Notebook Structure

| # | Section |
|---|---|
| 1 | Library Imports & Configuration |
| 2 | Data Loading & Initial Inspection |
| 3 | Data Cleaning |
| 4 | Exploratory Data Analysis (EDA) |
| 5 | Feature Engineering |
| 6 | Machine Learning – Model Training |
| 7 | Model Evaluation & Comparison |
| 8 | Feature Importance & Business Insights |
| 9 | Conclusion & Recommendations |

---
## Section 1 – Library Imports & Configuration

In [ ]:
# ── Core data manipulation ──────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Machine Learning – preprocessing ────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold

# ── Machine Learning – models ────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# ── Machine Learning – evaluation ────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

# ── Suppress warnings for clean output ──────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

# ── Global plot style ────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 13, 'axes.labelsize': 11})

print("✅ All libraries imported successfully.")
print(f"   NumPy  {np.__version__} | Pandas {pd.__version__}")

---
## Section 2 – Data Loading & Initial Inspection

In [ ]:
# ── Load the dataset ──────────────────────────────────────────────────────────
df = pd.read_csv('online food delivery dataset.csv')

# Strip any stray whitespace from column names
df.columns = df.columns.str.strip()

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
# Preview the first 5 rows
df.head()

In [ ]:
# Data types and non-null counts for every column
df.info()

In [ ]:
# Descriptive statistics for numerical columns
df.describe().round(2)

In [ ]:
# Descriptive statistics for categorical columns
df.describe(include='object')

---
## Section 3 – Data Cleaning

Steps performed:
1. Remove exact duplicate rows
2. Strip whitespace from all string values (the `Feedback` column has trailing spaces)
3. Identify and handle missing values
4. Detect and report outliers in numerical columns
5. Drop the unnamed trailing column that appears in the raw CSV

In [ ]:
# ── Step 1: Drop the unnamed last column (empty artefact from CSV export) ────
unnamed_cols = [c for c in df.columns if c.startswith('Unnamed')]
if unnamed_cols:
    df.drop(columns=unnamed_cols, inplace=True)
    print(f"Dropped unnamed columns: {unnamed_cols}")
else:
    print("No unnamed columns found.")

# ── Step 2: Strip whitespace from all object columns ────────────────────────
obj_cols = df.select_dtypes(include='object').columns
df[obj_cols] = df[obj_cols].apply(lambda col: col.str.strip())
print(f"\nStripped whitespace from {len(obj_cols)} text columns.")

# ── Step 3: Check for missing values ─────────────────────────────────────────
missing = df.isnull().sum()
print("\nMissing values per column:")
print(missing[missing > 0] if missing.any() else "  → No missing values detected. ✅")

In [ ]:
# ── Step 4: Duplicate rows ────────────────────────────────────────────────────
n_dupes = df.duplicated().sum()
print(f"Duplicate rows found: {n_dupes}")
if n_dupes > 0:
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"  → Dropped duplicates. New shape: {df.shape}")
else:
    print("  → No duplicates to remove. ✅")

In [ ]:
# ── Step 5: Outlier detection for numerical columns using IQR method ──────────
num_cols = ['Age', 'Family size', 'latitude', 'longitude', 'Pin code']

print("Outlier summary (IQR method):\n")
outlier_report = {}
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_report[col] = n_out
    print(f"  {col:15s}: {n_out:3d} potential outliers  (bounds [{lower:.2f}, {upper:.2f}])")

print("\nNote: latitude, longitude and Pin code outliers reflect genuine geographic spread.")
print("Age outliers (if any) will be reviewed below.")

In [ ]:
# ── Step 6: Value distributions for key categorical features ─────────────────
cat_review = ['Gender', 'Marital Status', 'Occupation', 'Monthly Income',
              'Educational Qualifications', 'Customer Type', 'Output', 'Feedback']

for col in cat_review:
    print(f"\n{col}:")
    print(df[col].value_counts().to_string())

In [ ]:
# ── Summary after cleaning ────────────────────────────────────────────────────
print("=" * 45)
print("CLEANED DATASET SUMMARY")
print("=" * 45)
print(f"  Rows    : {df.shape[0]}")
print(f"  Columns : {df.shape[1]}")
print(f"  Missing : {df.isnull().sum().sum()}")
print("=" * 45)

---
## Section 4 – Exploratory Data Analysis (EDA)

We explore:
- Target variable distribution
- Demographic breakdowns (Age, Gender, Marital Status, Occupation)
- Income and Education analysis
- Family size and Customer Type
- Feedback sentiment
- Correlation heatmap
- Geographic distribution

In [ ]:
# ── 4.1  Target Variable Distribution ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Target Variable – Orders Food Online (Output)', fontsize=14, fontweight='bold')

counts = df['Output'].value_counts()
colors = ['#2ecc71', '#e74c3c']

# Bar chart
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.2)
axes[0].set_xlabel('Output')
axes[0].set_ylabel('Count')
axes[0].set_title('Counts')
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 3, str(v), ha='center', fontsize=11)

# Pie chart
axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Proportion')

plt.tight_layout()
plt.show()

print(f"\nClass distribution:\n{counts.to_string()}")
print(f"\nClass balance ratio (Yes:No) = {counts['Yes']/counts['No']:.2f}:1")

In [ ]:
# ── 4.2  Age Distribution ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Age Distribution', fontsize=14, fontweight='bold')

# Overall histogram
axes[0].hist(df['Age'], bins=20, color='steelblue', edgecolor='white', linewidth=0.8)
axes[0].axvline(df['Age'].mean(), color='crimson', linestyle='--', linewidth=1.8, label=f"Mean = {df['Age'].mean():.1f}")
axes[0].axvline(df['Age'].median(), color='darkorange', linestyle='--', linewidth=1.8, label=f"Median = {df['Age'].median():.1f}")
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Age Histogram')
axes[0].legend()

# Age vs Output
df.boxplot(column='Age', by='Output', ax=axes[1],
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='crimson', linewidth=2))
axes[1].set_title('Age by Output')
axes[1].set_xlabel('Output (Orders Online)')
axes[1].set_ylabel('Age')
plt.suptitle('')  # remove pandas auto-suptitle

plt.tight_layout()
plt.show()

print(df.groupby('Output')['Age'].describe().round(2))

In [ ]:
# ── 4.3  Gender & Marital Status ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Gender & Marital Status vs Online Ordering', fontsize=14, fontweight='bold')

for ax, col, title in zip(axes,
                           ['Gender', 'Marital Status'],
                           ['Gender vs Output', 'Marital Status vs Output']):
    ct = pd.crosstab(df[col], df['Output'], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white', linewidth=0.8, rot=30)
    ax.set_title(title)
    ax.set_ylabel('% of Group')
    ax.set_xlabel('')
    ax.legend(title='Output', loc='upper right')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.show()

In [ ]:
# ── 4.4  Occupation ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Occupation Analysis', fontsize=14, fontweight='bold')

# Count of each occupation
occ_counts = df['Occupation'].value_counts()
axes[0].barh(occ_counts.index, occ_counts.values,
             color=sns.color_palette('Set2', len(occ_counts)), edgecolor='white')
axes[0].set_title('Occupation Counts')
axes[0].set_xlabel('Number of Customers')
for i, v in enumerate(occ_counts.values):
    axes[0].text(v + 1, i, str(v), va='center', fontsize=10)

# Occupation vs Output (% Yes)
occ_yes = df.groupby('Occupation')['Output'].apply(lambda x: (x == 'Yes').mean() * 100).sort_values()
axes[1].barh(occ_yes.index, occ_yes.values,
             color=sns.color_palette('RdYlGn', len(occ_yes)), edgecolor='white')
axes[1].set_title('% Who Order Online by Occupation')
axes[1].set_xlabel('% Ordering Online')
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter())
for i, v in enumerate(occ_yes.values):
    axes[1].text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ── 4.5  Monthly Income ───────────────────────────────────────────────────────
# Define ordered income levels for clean axis ordering
income_order = ['No Income', 'Below Rs.10000', '10001 to 25000',
                '25001 to 50000', 'More than 50000']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Monthly Income Analysis', fontsize=14, fontweight='bold')

# Distribution
income_counts = df['Monthly Income'].value_counts().reindex(income_order, fill_value=0)
axes[0].bar(income_counts.index, income_counts.values,
            color=sns.color_palette('Blues_d', len(income_order)), edgecolor='white')
axes[0].set_title('Income Distribution')
axes[0].set_xlabel('Income Bracket')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(income_counts.index, rotation=35, ha='right')
for i, v in enumerate(income_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=10)

# % Ordering online per income bracket
income_yes = (df.groupby('Monthly Income')['Output']
               .apply(lambda x: (x == 'Yes').mean() * 100)
               .reindex(income_order, fill_value=0))
colors_rg = sns.color_palette('RdYlGn', len(income_order))
axes[1].bar(income_yes.index, income_yes.values, color=colors_rg, edgecolor='white')
axes[1].set_title('% Ordering Online by Income')
axes[1].set_xlabel('Income Bracket')
axes[1].set_ylabel('% Ordering Online')
axes[1].set_xticklabels(income_yes.index, rotation=35, ha='right')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
for i, v in enumerate(income_yes.values):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── 4.6  Educational Qualifications ──────────────────────────────────────────
edu_order = ['Uneducated', 'School', 'Graduate', 'Post Graduate', 'Ph.D']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Education Level Analysis', fontsize=14, fontweight='bold')

edu_counts = df['Educational Qualifications'].value_counts().reindex(edu_order, fill_value=0)
axes[0].bar(edu_counts.index, edu_counts.values,
            color=sns.color_palette('Purples_d', len(edu_order)), edgecolor='white')
axes[0].set_title('Education Distribution')
axes[0].set_xlabel('Qualification')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(edu_counts.index, rotation=25, ha='right')
for i, v in enumerate(edu_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=10)

edu_yes = (df.groupby('Educational Qualifications')['Output']
             .apply(lambda x: (x == 'Yes').mean() * 100)
             .reindex(edu_order, fill_value=0))
axes[1].bar(edu_yes.index, edu_yes.values,
            color=sns.color_palette('RdYlGn', len(edu_order)), edgecolor='white')
axes[1].set_title('% Ordering Online by Education')
axes[1].set_xlabel('Qualification')
axes[1].set_ylabel('% Ordering Online')
axes[1].set_xticklabels(edu_yes.index, rotation=25, ha='right')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
for i, v in enumerate(edu_yes.values):
    axes[1].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── 4.7  Family Size & Customer Type ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Family Size & Customer Type', fontsize=14, fontweight='bold')

# Family size vs Output
fam_yes = (df.groupby('Family size')['Output']
             .apply(lambda x: (x == 'Yes').mean() * 100)
             .sort_index())
axes[0].bar(fam_yes.index.astype(str), fam_yes.values,
            color=sns.color_palette('Set2', len(fam_yes)), edgecolor='white')
axes[0].set_title('% Ordering Online by Family Size')
axes[0].set_xlabel('Family Size')
axes[0].set_ylabel('% Ordering Online')
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter())
for i, v in enumerate(fam_yes.values):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=10)

# Customer Type distribution
ctype = df['Customer Type'].value_counts()
axes[1].pie(ctype.values, labels=ctype.index,
            colors=sns.color_palette('Set2', len(ctype)),
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Customer Type Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# ── 4.8  Feedback Sentiment ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Customer Feedback Analysis', fontsize=14, fontweight='bold')

# Overall feedback split
fb_counts = df['Feedback'].value_counts()
fb_colors = ['#27ae60', '#e74c3c']
axes[0].bar(fb_counts.index, fb_counts.values, color=fb_colors, edgecolor='white')
axes[0].set_title('Overall Feedback')
axes[0].set_ylabel('Count')
for bar, v in zip(axes[0].patches, fb_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 2, str(v), ha='center', fontsize=11)

# Feedback vs Output
ct = pd.crosstab(df['Feedback'], df['Output'], normalize='index') * 100
ct.plot(kind='bar', ax=axes[1], color=fb_colors, edgecolor='white', rot=0)
axes[1].set_title('Feedback vs Output')
axes[1].set_ylabel('% of Feedback Group')
axes[1].set_xlabel('Feedback')
axes[1].legend(title='Output')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.show()

print("\nFeedback × Output crosstab (counts):")
print(pd.crosstab(df['Feedback'], df['Output']))

In [ ]:
# ── 4.9  Correlation Heatmap (numerical features) ────────────────────────────

# Encode Output numerically for correlation purposes (temporary)
df_corr = df.copy()
df_corr['Output_num'] = (df_corr['Output'] == 'Yes').astype(int)

corr_cols = ['Age', 'Family size', 'latitude', 'longitude', 'Output_num']
corr_matrix = df_corr[corr_cols].corr()

plt.figure(figsize=(7, 5))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, vmin=-1, vmax=1,
            annot_kws={'size': 11})
plt.title('Correlation Heatmap (Numerical Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.10  Geographic Distribution (Bengaluru PIN Codes) ──────────────────────
plt.figure(figsize=(8, 6))

yes_mask = df['Output'] == 'Yes'
plt.scatter(df.loc[yes_mask, 'longitude'], df.loc[yes_mask, 'latitude'],
            alpha=0.5, s=30, c='#2ecc71', label='Orders Online (Yes)', edgecolors='none')
plt.scatter(df.loc[~yes_mask, 'longitude'], df.loc[~yes_mask, 'latitude'],
            alpha=0.6, s=30, c='#e74c3c', label='Does Not Order (No)', edgecolors='none')

plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Customer Geographic Distribution – Bengaluru', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.11  Pair Plot (key numerical features coloured by Output) ───────────────
pair_df = df[['Age', 'Family size', 'latitude', 'longitude', 'Output']].copy()

g = sns.pairplot(pair_df, hue='Output', palette={'Yes': '#2ecc71', 'No': '#e74c3c'},
                 plot_kws={'alpha': 0.5, 's': 25},
                 diag_kind='kde')
g.fig.suptitle('Pair Plot – Key Features by Output', y=1.02, fontsize=13, fontweight='bold')
plt.show()

---
## Section 5 – Feature Engineering

Steps:
1. Encode ordinal features (income, education) with meaningful numeric levels
2. Label-encode binary/nominal categories
3. Create new derived features (age group, family category, income-has-income flag)
4. Drop columns not useful for modelling (coordinates, pin code, last-column)
5. Prepare final feature matrix `X` and target vector `y`

In [ ]:
# ── Work on a copy to preserve the original cleaned dataframe ─────────────────
df_ml = df.copy()

# ── 5.1  Ordinal encoding – Monthly Income ────────────────────────────────────
income_map = {
    'No Income'       : 0,
    'Below Rs.10000'  : 1,
    '10001 to 25000'  : 2,
    '25001 to 50000'  : 3,
    'More than 50000' : 4
}
df_ml['Income_Level'] = df_ml['Monthly Income'].map(income_map)
print("Income_Level distribution:")
print(df_ml['Income_Level'].value_counts().sort_index())

In [ ]:
# ── 5.2  Ordinal encoding – Education ─────────────────────────────────────────
edu_map = {
    'Uneducated'   : 0,
    'School'       : 1,
    'Graduate'     : 2,
    'Post Graduate': 3,
    'Ph.D'         : 4
}
df_ml['Edu_Level'] = df_ml['Educational Qualifications'].map(edu_map)
print("Edu_Level distribution:")
print(df_ml['Edu_Level'].value_counts().sort_index())

In [ ]:
# ── 5.3  Label Encoding – Binary / Low-cardinality nominal features ───────────
le = LabelEncoder()

# Gender: Female=0, Male=1 (Prefer not to say → 2)
df_ml['Gender_enc'] = le.fit_transform(df_ml['Gender'])

# Marital Status: Married=0, Prefer not to say=1, Single=2
df_ml['Marital_enc'] = le.fit_transform(df_ml['Marital Status'])

# Occupation: Employee=0, House wife=1, Self Employeed=2, Student=3
df_ml['Occ_enc'] = le.fit_transform(df_ml['Occupation'])

# Customer Type: Frequent=0, New=1, Regular=2
df_ml['CType_enc'] = le.fit_transform(df_ml['Customer Type'])

# Feedback: Negative=0, Positive=1
df_ml['Feedback_enc'] = le.fit_transform(df_ml['Feedback'])

# Target: No=0, Yes=1
df_ml['Target'] = (df_ml['Output'] == 'Yes').astype(int)

print("Encoding complete. New encoded columns:")
print(['Gender_enc','Marital_enc','Occ_enc','CType_enc','Feedback_enc','Target'])

In [ ]:
# ── 5.4  Derived Features ─────────────────────────────────────────────────────

# Age Group bins
bins = [0, 21, 25, 30, 100]
labels = ['18-21', '22-25', '26-30', '31+']
df_ml['Age_Group'] = pd.cut(df_ml['Age'], bins=bins, labels=labels, right=True)
df_ml['Age_Group_enc'] = df_ml['Age_Group'].cat.codes   # ordinal 0-3

# Family Category
df_ml['Family_Cat'] = pd.cut(df_ml['Family size'], bins=[0,1,3,6],
                              labels=['Solo', 'Small', 'Large'])
df_ml['Family_Cat_enc'] = df_ml['Family_Cat'].cat.codes  # ordinal 0-2

# Has Income flag
df_ml['Has_Income'] = (df_ml['Income_Level'] > 0).astype(int)

# Positive Feedback flag
df_ml['Pos_Feedback'] = df_ml['Feedback_enc']   # already 0/1

print("Derived features created:")
print(df_ml[['Age_Group','Age_Group_enc','Family_Cat','Family_Cat_enc',
             'Has_Income','Pos_Feedback']].head(10))

In [ ]:
# ── 5.5  Assemble Final Feature Matrix ───────────────────────────────────────
FEATURES = [
    'Age',
    'Gender_enc',
    'Marital_enc',
    'Occ_enc',
    'Income_Level',
    'Edu_Level',
    'Family size',
    'CType_enc',
    'Feedback_enc',
    'Age_Group_enc',
    'Family_Cat_enc',
    'Has_Income',
]

TARGET = 'Target'

X = df_ml[FEATURES]
y = df_ml[TARGET]

print(f"Feature matrix shape : {X.shape}")
print(f"Target vector shape  : {y.shape}")
print(f"\nClass distribution   : {dict(y.value_counts())}")
print(f"\nFeatures used:\n{FEATURES}")

In [ ]:
# ── 5.6  Train / Test Split (80:20, stratified) ───────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set : {X_train.shape[0]} samples")
print(f"Test set     : {X_test.shape[0]} samples")
print(f"\nTrain class balance – 1 (Yes): {y_train.mean()*100:.1f}%")
print(f"Test  class balance – 1 (Yes): {y_test.mean()*100:.1f}%")

In [ ]:
# ── 5.7  Feature Scaling (for distance/margin-based models) ──────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("StandardScaler fit on training data. Scaled arrays ready for SVM and KNN.")

---
## Section 6 – Machine Learning Model Training

Six classifiers are trained and compared:

| Model | Key Characteristic |
|---|---|
| Logistic Regression | Linear baseline, interpretable coefficients |
| Decision Tree | Tree-based, highly interpretable |
| Random Forest | Ensemble bagging, robust to overfitting |
| Gradient Boosting | Sequential ensemble, usually top performer |
| SVM (RBF) | Margin maximisation, good on small datasets |
| K-Nearest Neighbours | Instance-based, non-parametric |

In [ ]:
# ── Instantiate all models ────────────────────────────────────────────────────
models = {
    'Logistic Regression'  : LogisticRegression(max_iter=500, random_state=42),
    'Decision Tree'        : DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest'        : RandomForestClassifier(n_estimators=200, max_depth=8,
                                                    random_state=42, n_jobs=-1),
    'Gradient Boosting'    : GradientBoostingClassifier(n_estimators=200, learning_rate=0.08,
                                                        max_depth=4, random_state=42),
    'SVM (RBF)'            : SVC(kernel='rbf', probability=True, random_state=42),
    'KNN'                  : KNeighborsClassifier(n_neighbors=7),
}

# ── 5-fold stratified CV on TRAINING data (unscaled for tree models) ─────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
print("Cross-Validation Accuracy (5-fold) on Training Set")
print("-" * 55)

for name, model in models.items():
    # Use scaled data for SVM and KNN
    X_cv = X_train_sc if name in ('SVM (RBF)', 'KNN') else X_train
    scores = cross_val_score(model, X_cv, y_train, cv=cv, scoring='accuracy')
    cv_results[name] = scores
    print(f"  {name:25s}: {scores.mean()*100:5.2f}% ± {scores.std()*100:.2f}%")

In [ ]:
# ── Train all models on the full training set ─────────────────────────────────
trained = {}
for name, model in models.items():
    X_tr = X_train_sc if name in ('SVM (RBF)', 'KNN') else X_train
    model.fit(X_tr, y_train)
    trained[name] = model

print("All models trained. ✅")

---
## Section 7 – Model Evaluation & Comparison

In [ ]:
# ── 7.1  Consolidated Test-Set Metrics ───────────────────────────────────────
from sklearn.metrics import precision_score, recall_score, f1_score

results = []
for name, model in trained.items():
    X_te = X_test_sc if name in ('SVM (RBF)', 'KNN') else X_test
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]

    results.append({
        'Model'    : name,
        'Accuracy' : accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall'   : recall_score(y_test, y_pred, zero_division=0),
        'F1 Score' : f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC'  : roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
results_df = results_df.set_index('Model')

# Pretty-print with percentage formatting
print("\n" + "="*70)
print("MODEL COMPARISON – TEST SET PERFORMANCE")
print("="*70)
print(results_df.applymap(lambda v: f'{v*100:.2f}%').to_string())
print("="*70)
print(f"\n🏆 Best model (ROC-AUC): {results_df['ROC-AUC'].idxmax()}")

In [ ]:
# ── 7.2  Metric Comparison Bar Chart ─────────────────────────────────────────
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
plot_df = results_df[metrics].reset_index()

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(plot_df))
width = 0.14
palette = sns.color_palette('Set2', len(metrics))

for i, (metric, color) in enumerate(zip(metrics, palette)):
    offset = (i - len(metrics)/2 + 0.5) * width
    bars = ax.bar(x + offset, plot_df[metric] * 100, width,
                  label=metric, color=color, edgecolor='white', linewidth=0.6)

ax.set_xticks(x)
ax.set_xticklabels(plot_df['Model'], rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Score (%)')
ax.set_title('Model Comparison – All Metrics', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.set_ylim(0, 110)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.show()

In [ ]:
# ── 7.3  ROC Curves (all models) ─────────────────────────────────────────────
plt.figure(figsize=(8, 6))
palette_roc = sns.color_palette('tab10', len(trained))

for (name, model), color in zip(trained.items(), palette_roc):
    X_te = X_test_sc if name in ('SVM (RBF)', 'KNN') else X_test
    y_prob = model.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, linewidth=1.8, color=color, label=f'{name} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves – All Models', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=8.5)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.4  Confusion Matrices (all 6 models) ────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Confusion Matrices – Test Set', fontsize=14, fontweight='bold')

for ax, (name, model) in zip(axes.flat, trained.items()):
    X_te = X_test_sc if name in ('SVM (RBF)', 'KNN') else X_test
    y_pred = model.predict(X_te)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No', 'Yes'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    acc = accuracy_score(y_test, y_pred)
    ax.set_title(f'{name}\n(Acc={acc*100:.1f}%)', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ── 7.5  Detailed Classification Report – Best Model (Random Forest) ──────────
best_name = results_df['ROC-AUC'].idxmax()
best_model = trained[best_name]
X_te_best = X_test_sc if best_name in ('SVM (RBF)', 'KNN') else X_test
y_pred_best = best_model.predict(X_te_best)

print(f"Classification Report – {best_name}")
print("=" * 55)
print(classification_report(y_test, y_pred_best, target_names=['No', 'Yes']))

In [ ]:
# ── 7.6  Cross-Validation Score Distribution Boxplot ─────────────────────────
cv_df = pd.DataFrame(cv_results) * 100

plt.figure(figsize=(10, 5))
cv_df.boxplot(vert=False,
              boxprops=dict(color='steelblue'),
              medianprops=dict(color='crimson', linewidth=2),
              whiskerprops=dict(color='steelblue'),
              capprops=dict(color='steelblue'),
              flierprops=dict(marker='o', color='grey', markersize=5))
plt.xlabel('Accuracy (%)')
plt.title('5-Fold CV Accuracy Distribution per Model', fontsize=13, fontweight='bold')
plt.xaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.show()

---
## Section 8 – Feature Importance & Business Insights

In [ ]:
# ── 8.1  Random Forest – Feature Importance ──────────────────────────────────
rf_model = trained['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=FEATURES)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(9, 5))
colors_fi = sns.color_palette('RdYlGn', len(importances))
bars = plt.barh(importances.index, importances.values * 100,
                color=colors_fi, edgecolor='white', linewidth=0.7)
plt.xlabel('Importance (%)')
plt.title('Random Forest – Feature Importance', fontsize=13, fontweight='bold')
for bar, v in zip(bars, importances.values * 100):
    plt.text(v + 0.3, bar.get_y() + bar.get_height()/2,
             f'{v:.2f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print("\nTop 5 Features:")
print(importances.tail(5).sort_values(ascending=False).apply(lambda v: f'{v*100:.2f}%').to_string())

In [ ]:
# ── 8.2  Gradient Boosting – Feature Importance ───────────────────────────────
gb_model = trained['Gradient Boosting']
gb_imp = pd.Series(gb_model.feature_importances_, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
colors_gb = sns.color_palette('Blues_d', len(gb_imp))
plt.barh(gb_imp.index, gb_imp.values * 100, color=colors_gb, edgecolor='white')
plt.xlabel('Importance (%)')
plt.title('Gradient Boosting – Feature Importance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.3  Logistic Regression – Coefficient Plot ───────────────────────────────
lr_model = trained['Logistic Regression']
coef = pd.Series(lr_model.coef_[0], index=FEATURES).sort_values()

plt.figure(figsize=(9, 5))
colors_lr = ['#e74c3c' if v < 0 else '#27ae60' for v in coef.values]
plt.barh(coef.index, coef.values, color=colors_lr, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.9)
plt.xlabel('Coefficient Value')
plt.title('Logistic Regression – Feature Coefficients', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nPositive coefficients → increase probability of ordering online")
print("Negative coefficients → decrease probability of ordering online")

In [ ]:
# ── 8.4  Business Insight Summaries ──────────────────────────────────────────

print("━" * 60)
print("BUSINESS INSIGHTS FROM EDA & FEATURE IMPORTANCE")
print("━" * 60)

insights = [
    ("Feedback",
     "Positive feedback is the single strongest predictor of "
     "online ordering behaviour. Investing in service quality "
     "and satisfaction drives repeat orders."),

    ("Customer Type (Frequent)",
     "Frequent customers have a disproportionately high "
     "conversion to online ordering. Loyalty programmes "
     "targeting this segment yield the highest ROI."),

    ("Family Size",
     "Larger families (4+ members) show higher online ordering "
     "rates. Bundle/combo deals and group discounts are "
     "well-targeted at these households."),

    ("Income Level",
     "Mid-to-high income brackets (₹10,000–50,000) order "
     "online more, though 'No Income' students are also "
     "significant due to volume. Student promotions remain "
     "important."),

    ("Age",
     "The core ordering demographic is 21–27 years. Marketing "
     "should prioritise digital channels (Instagram, Swiggy, "
     "Zomato app) favoured by this age group."),

    ("Occupation",
     "Students dominate the dataset but Employees and "
     "Self-Employed individuals have higher per-capita spend. "
     "Premium tiers or subscription models suit working "
     "professionals."),

    ("Education",
     "Post-Graduate and Graduate customers form the majority "
     "of users. Digital literacy correlates with comfort in "
     "online ordering; uneducated segments need UX simplification."),
]

for i, (topic, text) in enumerate(insights, 1):
    print(f"\n  {i}. [{topic}]")
    # Word-wrap to 55 chars
    import textwrap
    wrapped = textwrap.fill(text, width=55)
    for line in wrapped.split('\n'):
        print(f"     {line}")

print("\n" + "━" * 60)

---
## Section 9 – Conclusion & Recommendations

### 9.1  Project Summary

This notebook delivered an end-to-end analytics pipeline on the **Online Food Delivery dataset (388 records, 14 features)** from Bengaluru, India. 

Key steps completed:

| Stage | Key Finding |
|---|---|
| Data Cleaning | Trailing whitespace in `Feedback`, no nulls, duplicate rows removed |
| EDA | ~75% of customers order online; students & young adults dominate |
| Feature Engineering | 12 features used; ordinal encoding for income & education |
| Best ML Model | See cell below for final result |
| Top Predictors | Feedback, Customer Type, Family Size, Income Level, Age |

### 9.2  Business Recommendations

1. **Prioritise feedback loops** – Positive feedback is the #1 predictor. Implement post-delivery rating nudges.
2. **Frequent-customer retention** – Create a tiered loyalty programme (Bronze → Gold).
3. **Family bundling** – Offer combo meals and group discounts targeting families of 4–6.
4. **Student outreach** – Run campus ambassador programmes and student-exclusive discounts.
5. **Income-aware pricing** – Introduce entry-level meal options for the 'No Income' segment without cannibalising premium orders.
6. **Geographic micro-targeting** – Deploy location-specific promotions using the PIN code clusters identified in EDA.

### 9.3  Model Deployment Path

The best-performing model can be:
- **Serialised** via `joblib` and deployed as a REST API (Flask/FastAPI)
- **Integrated** into the ordering platform to score new sign-ups in real time
- **Retrained** monthly as new order data accumulates

---
> **IBM SkillsBuild Internship** | FoodWise AI Analytics | Notebook complete ✅

In [ ]:
# ── Final Model Summary ───────────────────────────────────────────────────────
print("╔" + "═"*60 + "╗")
print("║" + " FoodWise AI – Final Model Performance Summary ".center(60) + "║")
print("╠" + "═"*60 + "╣")

for _, row in results_df.reset_index().iterrows():
    line = f"  {row['Model']:<26} | AUC: {row['ROC-AUC']*100:.1f}%  Acc: {row['Accuracy']*100:.1f}%  F1: {row['F1 Score']*100:.1f}%"
    print("║" + line.ljust(60) + "║")

print("╠" + "═"*60 + "╣")
best = results_df['ROC-AUC'].idxmax()
best_acc = results_df.loc[best, 'Accuracy'] * 100
best_auc = results_df.loc[best, 'ROC-AUC'] * 100
winner = f"  🏆 Winner: {best}  |  Acc: {best_acc:.1f}%  AUC: {best_auc:.1f}%"
print("║" + winner.ljust(60) + "║")
print("╚" + "═"*60 + "╝")

print("\n✅ FoodWise AI Notebook – IBM SkillsBuild Internship – COMPLETE")

In [ ]:
# ── Optional: Save the best model to disk ─────────────────────────────────────
import joblib

best_name = results_df['ROC-AUC'].idxmax()
best_model = trained[best_name]

model_filename = 'foodwise_best_model.pkl'
joblib.dump(best_model, model_filename)

print(f"Model '{best_name}' saved to '{model_filename}'")
print("Load it later with: model = joblib.load('foodwise_best_model.pkl')")